<a href="https://colab.research.google.com/github/wonder4hth/Detect-Abnormal-Behavior-in-Autonomous-Driving/blob/main/02_mon_exploring_transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transformers, Tasks, and Model Heads

**Learning Outcomes**:
* Understand that LLMs are just one single task-based architecture of transformers
* Become convinced that it's not just magic - the models we use are truly what was written in the Attention is All You Need Paper
* Training a model for a specific task may be a better option than a huge, online LLM (but also maybe not!)
* Introduction to Huggingface 🤗

***


<font color="blue"><strong>Strong suggestion: if you're using Google Colab, try to choose a GPU runtime to accelerate the code below. Otherwise, you will have graduated and already be employed by the time some of this code runs. </strong></font>
<font color="blue"><strong> ✨ You're welcome. ✨ </strong></p>

In [1]:
!pip install -q torchinfo

## Transformers In Code


### The Pipeline from Start to Finish

![The Transformer Architecture](https://huggingface.co/datasets/huggingface-course/documentation-images/resolve/main/en/chapter2/full_nlp_pipeline.svg)
Reference: [Huggingface LLM Course, Chapter 2.2](https://huggingface.co/learn/llm-course/chapter2/2)

In [2]:
from transformers import pipeline
import torch
import textwrap

In [3]:
# download ultra small model
llm = pipeline("text-generation", model="gpt2")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [4]:
#checkout elements of a full pipeline above
llm.tokenizer


GPT2Tokenizer(name_or_path='gpt2', vocab_size=50257, model_max_length=1024, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>'}, added_tokens_decoder={
	50256: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
})

In [7]:
# model with task-specific head
llm.model
llm.model.lm_head

Linear(in_features=768, out_features=50257, bias=False)

Let's generate some text through the entire pipeline, where the  post-processing and iteration is done behind the scenes

In [8]:
# get the response, noting that we can change the length of the response if we want something longer
response = llm("The cat's paw hovered menacingly above the precious glass cup precariously positioned on the counter. His eyes",
               max_length=100, temperature=0.8)

# get the actual response out of the formatted output
print(textwrap.fill(response[-1]['generated_text']))

The cat's paw hovered menacingly above the precious glass cup
precariously positioned on the counter. His eyes glinted. Her face was
dark, but it was still the same black and red. She was staring at him
in his dark eyes. His hands were trembling. And her mouth was
trembling. "We have to stay here," he said to her, "because we have to
protect you."  "No," she said softly, "I'm not going to leave anything


Try the following with the instruct model to see the difference in behavior.

In [ ]:
#text = "Write a haiku about a very mischevious cat."

#format it correctly
#convo_message = [{'role':'user', "content":text}]

# pass input to model and get a response
#llm_response = llm(convo_message, max_length=100)

# get response only
#response = llm_response[0]['generated_text'][-1]['content']
#print(response)


---



## Attention is All You Need (Vaswani, 2019)

Peeling back another layer, let's now start to focus on the transformer model alone without the rest of the infrastructure with the tokenizer. We can see the transformer body which enriches token representations then connected to a task-specific head. We'll look more into the architectures and the task-specific heads as well.

<div style="display:flex; justify-content:center; align-items:center;">

  <figure style="margin:0; text-align:center;max-width:70%;">
    <img src="https://huggingface.co/datasets/huggingface-course/documentation-images/resolve/main/en/chapter2/transformer_and_head.svg" alt="Figure 1" style="max-width:80%; height:auto;">
    <figcaption style="margin-top:8px;">
    Proposed foundational encoder-decoder transformers architecture
    </figcaption>
  </figure>

  <figure style="margin:0; text-align:left;">
    <p> One of many architectures:</p>
    <ul>
        <li>Encoder Only</li>
        <li>Decoder Only</li>
        <li>Encoder + Decoder</li>
    </ul>
    <p> Older style transformers, but applicable for advanced stages of cost reduction dependent on task. </p>
  </figure>

</div>

### Encoder Architectures


In [9]:
import torch
from torchinfo import summary

#transformer imports
from transformers import AutoModel, BertModel

In [10]:
# Load BERT and let's just look at it
bert = AutoModel.from_pretrained("bert-base-uncased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
### We can also look at some of the listed layers
# in the model (though they may be out of order)


We can learn about the model parameters and some configuration...

And the explicit layout of the model...

In [11]:
dummy_input = torch.randint(0, 1000, (1, 16))

# Summarize the architectural hierarchy
summary(bert, input_data=dummy_input,
    depth=10,                           # Expands down to Self-Attention and FC layers
    col_names=["num_params",
               #"trainable",
               "input_size",
               "output_size"],
    row_settings=["var_names"]
)

Layer (type (var_name))                                           Param #                   Input Shape               Output Shape
BertModel (BertModel)                                             --                        [1, 16]                   [1, 768]
├─BertEmbeddings (embeddings)                                     --                        --                        [1, 16, 768]
│    └─Embedding (word_embeddings)                                23,440,896                [1, 16]                   [1, 16, 768]
│    └─Embedding (token_type_embeddings)                          1,536                     [1, 16]                   [1, 16, 768]
│    └─Embedding (position_embeddings)                            393,216                   [1, 16]                   [1, 16, 768]
│    └─LayerNorm (LayerNorm)                                      1,536                     [1, 16, 768]              [1, 16, 768]
│    └─Dropout (dropout)                                          --                   

### Encoder-Decoder
Lets check out t5 by Google

In [12]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

In [13]:
#Add model name from HF for t5
encoder_decoder_mdl_name = "google-t5/t5-small"

In [14]:
t5 = AutoModelForSeq2SeqLM.from_pretrained(encoder_decoder_mdl_name)
t5_tokenizer = AutoTokenizer.from_pretrained(encoder_decoder_mdl_name)

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [ ]:
# look at it


In [17]:
#try it for translation
en = "Where is the library?"

# prepare the inputs using the tokenizer
inputs = t5_tokenizer(en, return_tensors="pt")
inputs

# generate the translation
response = t5.generate(**inputs, max_new_tokens=50)

# decode the response
t5_tokenizer.decode(response[0], skip_special_tokens=True)

# get the response


# decoder the response


'Wo ist die Bibliothek?'

In [ ]:
#look at inputs to see the actual behavior of the tokenizer
#inputs

### Decoder Only

**Try it Yourself!**

<font color="blue">We've looked at encoder models and encoder-decoder models, but what about decoder only models. Try to look at this model on your own and generate a response from the model. Make sure that you look at the architecture using `torchinfo` and also generate a response. You can either use the model `gpt2` or `huggingfacetb/smollm2-135m` that we already downloaded above, except you shouldn't use the pipeline directly. </font>


<details>
<summary>Click here if you want to see some of the code</summary>
<p>Though of course generative AI can help.</p>
<pre>
##Imports for GPT2
from transformers import AutoModel, GPT2Model <br>
##Exploring GPT2
gpt2 = GPT2Model.from_pretrained("gpt2")
gpt2
</pre>
</details>

<br>

---
<br>


## The Transformer: Body + Head



In [ ]:
# First, let's install what we need (uncomment if needed)
# !pip install transformers datasets torch accelerate

In [ ]:
from transformers import BertModel, BertForSequenceClassification, BertForMaskedLM

In [ ]:
# Base BERT - no head, just the transformer
bert_base = BertModel.from_pretrained("bert-base-uncased")
bert_base

In [ ]:
# BERT for classification (e.g., sentiment analysis)
bert_classifier = BertForSequenceClassification.from_pretrained("bert-base-uncased")
bert_classifier

In [ ]:
# Let's see just the classifier head
bert_classifier.classifier

In [ ]:
# How many new parameters did the head add?
base_params = sum(p.numel() for p in bert_base.parameters())
classifier_params = sum(p.numel() for p in bert_classifier.parameters())
head_params = classifier_params - base_params
print(f"Base model with: {base_params:,} parameters")
print(f"With classifier head: {classifier_params:,} parameters")
print(f"Head added: {head_params:,} parameters")
print(f"That's only {head_params / classifier_params * 100:.2f}% of the total!")

### Token Classification Head
This head classifies each individual token, that is, for part of speech tagging and named entity recognition.

In [ ]:
from transformers import BertForTokenClassification

In [ ]:
# Load a token classification head
bert_token_classifier = BertForTokenClassification.from_pretrained("bert-base-uncased")
bert_token_classifier

In [ ]:
# This one has a classifier that applies to EACH token, not just [CLS]
bert_token_classifier.classifier

### Masked Language Modeling (MLM)

In [ ]:
from transformers import BertForMaskedLM, AutoModelForMaskedLM

In [ ]:
# Masked language model head - what BERT was originally trained with
bert_mlm = BertForMaskedLM.from_pretrained("bert-base-uncased")
bert_mlm

**Summary of heads we've seen:**

| Task | Head | Output Shape |
|------|------|--------------|
| Classification | `classifier` | (batch, num_classes) |
| Token Classification | `classifier` | (batch, seq_len, num_classes) |
| Question Answering | `qa_outputs` | start + end positions |
| Masked LM | `cls` with predictions | (batch, seq_len, vocab_size) |

### Huggingface, Models, and Tasks

Why does the download of the pretrained model with different architectures continue to give us errors?

-> The body is downloaded as the checkpoint since bert-base-uncased is an MLM model.

#### **Effect of Transfer Learning**

![The Transformer Architecture](https://huggingface.co/datasets/huggingface-course/documentation-images/resolve/main/en/chapter2/transformer_and_head.svg)
Reference: [Huggingface LLM Course, Chapter 2.2](https://huggingface.co/learn/llm-course/chapter2/2)

Let's try BERT! We know that it was trained on objectives other than classification, so let's see if the knowledge transfers to classification.

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

# 1. Load a larger subset of the dataset (5,000 rows instead of 500)
dataset = load_dataset("ai4privacy/pii-masking-200k", split="train[:5000]")
dataset = dataset.train_test_split(test_size=0.1) # Split for train/eval

# 2. Load tokenizer (load model later for easier restart)
model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)


In [ ]:
# 3. Preprocess the data
def preprocess_function(examples):
    prefix = "Mask PII in English: "
    # Prepend the prefix to each source text
    inputs = [prefix + text for text in examples["source_text"]]
    #inputs = examples["source_text"]
    targets = examples["target_text"]

    # Tokenize the source texts (with prefix)
    model_inputs = tokenizer(inputs, max_length=300, truncation=True)

    # Tokenize labels
    labels = tokenizer(targets, max_length=300, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset.map(preprocess_function, batched=True, remove_columns=dataset["train"].column_names)

In [ ]:
# load model
# model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [ ]:
# test model before training
test_text = ["Mask PII in English: Did Jane Anderson take the test today?",
             "Mask PII in English: Her SSN is 255-23-3454.",
             "Mask PII in English: They live at 404 Fullbrook Ln."]

def test_model(test_text):
  model.eval()
  test_inputs = tokenizer(test_text, return_tensors="pt", padding="longest").to(model.device)
  test_outputs = model.generate(**test_inputs, max_new_tokens=50)
  model.train()
  return tokenizer.decode(test_outputs, skip_special_tokens=True)

#actually test the model
#test_model(test_text)

In [ ]:
# 4. Data collator for dynamic padding
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# 5. Define training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./t5-small-pii-masking",
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=24,
    per_device_eval_batch_size=24,
    weight_decay=0.01,
    #save_total_limit=3,
    num_train_epochs=5,
    predict_with_generate=True,
    fp16=False, # T5 models often have unstable loss with fp16, disabling it
)

# 6. Initialize Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    #tokenizer=tokenizer,
    data_collator=data_collator,
)

# 7. Train the model
trainer.train()

In [ ]:
# look at the final behavior
test_model(test_text)